# Project name: Demand-Hype Divergence Score Prediction

## 📝 Project info:

#### 🏭 Industry  
- E-commerce & Marketing 

#### 🌐 Domain  
- Lifestyle & Consumer Themes  

## 📂 Dataset Information

#### 🔎 Sources
- **GDELT GKG** (Global Knowledge Graph)  
- **Google Trends**  
- **World Bank Open Data (WDI)**
- **Holidays(Python Library)**
#### All the datasets are **open source** and publicly available.  
- ✅ No privacy, licensing, or access restrictions  
- ✅ Safe for research and personal use 

#### Problem Name: Demand-Hype Divergence Score Prediction

#### ML Task: Regression

##### Problem Statement:
A key strategic question: is a market being over-covered by media relative to actual consumer demand, or is there an untapped demand opportunity? This model predicts a "Divergence Score" — a derived continuous target — that quantifies how much a market deviates from the expected demand-to-coverage balance, enabling media budget reallocation and editorial prioritization.

##### Why This Problem:
EDA showed Demand_to_Hype_Ratio ↔ Net_Sentiment | r = 0.316 and ↔ Activity_Density | r = -0.287 — sentiment and media activity partially explain demand-media divergence. This regression estimates why divergence happens and gives brands a forward-looking signal.

##### Expected ML Output:
A continuous Divergence_Score per country-category-week. Positive values flag under-served opportunities; negative values flag over-hyped markets.


In [1]:
import pandas as pd
import pymysql
import os
import sys
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import mysql.connector
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

### Connecting to TIDB

In [2]:
# Load .env file from the current directory
load_dotenv()
# Get connection details
DB_HOST = os.getenv('DB_HOST')
DB_PORT = int(os.getenv('DB_PORT', 4000))
DB_USERNAME = os.getenv('DB_USERNAME')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

# Verify all variables are loaded
print("\n=== Connection Details ===")
print(f"Host: {DB_HOST}")
print(f"Port: {DB_PORT}")
print(f"Username: {DB_USERNAME}")
print(f"Database: {DB_DATABASE}")

if not all([DB_HOST, DB_USERNAME, DB_PASSWORD, DB_DATABASE]):
    print("\n✗ ERROR: Some environment variables are missing!")
    print("Check your .env file has all required values")
    sys.exit(1)

print("\n✓ All variables loaded!")


=== Connection Details ===
Host: gateway01.ap-northeast-1.prod.aws.tidbcloud.com
Port: 4000
Username: qgNsbp7EdsPhLkz.root
Database: GDELT&TRENDS

✓ All variables loaded!


In [3]:
# Test connection
try:
    connection = pymysql.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USERNAME,
        password=DB_PASSWORD,
        database=DB_DATABASE,
        charset='utf8mb4'
    )
    
    with connection.cursor() as cursor:
        cursor.execute("SELECT 1")
        result = cursor.fetchone()
        print("✓ Successfully connected to TiDB Cloud!")
        if connection.open:
            query = "SELECT * FROM main;"
            df = pd.read_sql(query, connection)
            print("Totla Records:",len(df))
            print("Total Columns", len(df.columns.to_list()))
            display(df.columns.to_list())

    connection.close()
    
except pymysql.Error as e:
    print(f"✗ Connection Error: {e}")
    print(f"\nDebugging Info:")
    print(f"  Host: {DB_HOST}")
    print(f"  Port: {DB_PORT}")
    print(f"  User: {DB_USERNAME}")
    print(f"  Database: {DB_DATABASE}")

✓ Successfully connected to TiDB Cloud!
Totla Records: 8022
Total Columns 20


['week_start',
 'country_name',
 'category',
 'search_interest',
 'search_velocity',
 'search_acceleration',
 'media_volume',
 'tone_net_sentiment',
 'tone_positive_score',
 'tone_negative_score',
 'tone_polarity',
 'tone_activity_density',
 'tone_self_group_density',
 'source_diversity',
 'demand_to_hype_ratio',
 'inflation_rate',
 'internet_penetration',
 'gdp_per_capita',
 'is_holiday',
 'holiday_count']

In [4]:
df.columns[df.isnull().sum()>0]

Index([], dtype='object')

### 👩🏻‍💻 1. Modeling

#### 🛠️ 1.1 Pre-Requisites

### 🎯 1.1.2 X & y

#### Features (X-columns)

| Feature | Role |
| --- | --- |
| Country_Name | Segment context (categorical) |
| Category | Segment context (categorical) |
| Search_Velocity | Demand momentum (rate of change) |
| Search_Acceleration | Demand momentum (second derivative) |
| Media_Volume | Coverage intensity |
| tone_net_sentiment | Sentiment context |
| tone_polarity | Emotional tone of coverage |
| tone_activity_density | Event-driven coverage intensity |
| Source_Diversity | Coverage breadth |
| Inflation_Rate | Economic demand amplifier |
| Internet_Penetration | Media ecosystem capacity |
| holiday_count | Calendar demand spikes (numeric) |

---

#### Target (y-column)

| Target | Description |
| --- | --- |
| Divergence_Score | Regression target |

---

#### X-columns that need to be created (Feature Engineering)

| New Feature | Role |
| --- | --- |
| Search_Velocity_Lag1, Search_Velocity_Lag2 | Lagged demand momentum |
| Quarter | Seasonality (derived from Week_Start) |
| Month | Seasonality (derived from Week_Start) |

### Columns to Exclude

| Column | Reason for Exclusion |
| --- | --- |
| Country_Code | Redundant with Country_Name (categorical already included) |
| Search_Interest | Raw demand signal, replaced by velocity/acceleration features |
| tone_positive_score | Subsumed by tone_net_sentiment & tone_polarity |
| tone_negative_score | Subsumed by tone_net_sentiment & tone_polarity |
| tone_self_group_density | Not directly relevant to divergence modeling |
| Demand_to_Hype_Ratio | Derived metric, excluded to avoid leakage |
| Media_Volume_per_Source | Redundant with Media_Volume & Source_Diversity |
| Interest_per_Source | Redundant with Search_Velocity & Search_Acceleration |
| GDP_Per_Capita | Economic context, excluded to reduce collinearity |
| Year | Seasonality captured via Month/Quarter/Week_Start instead |
| is_holiday | Binary flag less informative than holiday_count |


#### ✂️ 1.1.2 Train-Test Split:

In [5]:
df['week_start'] = pd.to_datetime(df['week_start'])

In [6]:
# Extract month and year
df['year_month'] = df['week_start'].dt.to_period('M')

# Count how many weeks fall in each month
month_counts = df['year_month'].unique()

print(len(month_counts))
print(month_counts)


44
<PeriodArray>
['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07',
 '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02',
 '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09',
 '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04',
 '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11',
 '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06',
 '2026-07', '2026-08']
Length: 44, dtype: period[M]


In [7]:
# Find the last 6 unique months
last_6_months = df['year_month'].unique()[-6:]

In [8]:
# Split into train and test
train_df = df[~df['year_month'].isin(last_6_months)]
test_df  = df[df['year_month'].isin(last_6_months)]

print("Train months:", train_df['year_month'].unique())
print("Test months:", test_df['year_month'].unique())

Train months: <PeriodArray>
['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07',
 '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02',
 '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09',
 '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04',
 '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11',
 '2025-12', '2026-01', '2026-02']
Length: 38, dtype: period[M]
Test months: <PeriodArray>
['2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08']
Length: 6, dtype: period[M]


In [9]:
print("X_train:",train_df.shape,"\nX_test:",test_df.shape)

X_train: (6930, 21) 
X_test: (1092, 21)


### **Deriving y column from the dataframe**
### Divergence_Score = Search_Interest_norm - log_Media_Volume_norm

In [10]:
# log_Media_Volume = log1p(Media_Volume)
# Search_Interest_norm = (Search_Interest - Min_Category) / (Max_Category - Min_Category)
# log_Media_Volume_norm = (log_Media_Volume - Min_Category) / (Max_Category - Min_Category)

# since Media_Volume is having null values, we fill those null values with 0 and then procede with the calculation of log_Media_Volume and log_Media_Volume_norm
# Fill missing Media_Volume values with 0 for both sets
train_df['media_volume'] = train_df['media_volume'].fillna(0)
test_df['media_volume'] = test_df['media_volume'].fillna(0)

# Compute log_Media_Volume
train_df['log_media_volume'] = np.log1p(train_df['media_volume'])
test_df['log_media_volume'] = np.log1p(test_df['media_volume'])

# Calculate min and max strictly from the TRAINING set (prevents data leakage)
si_min = train_df['search_interest'].min()
si_max = train_df['search_interest'].max()

log_mv_min = train_df['log_media_volume'].min()
log_mv_max = train_df['log_media_volume'].max()

# Normalize Search_Interest using training stats
train_si_norm = (train_df['search_interest'] - si_min) / (si_max - si_min)
test_si_norm = (test_df['search_interest'] - si_min) / (si_max - si_min)

# Normalize log_Media_Volume using training stats
train_log_mv_norm = (train_df['log_media_volume'] - log_mv_min) / (log_mv_max - log_mv_min)
test_log_mv_norm = (test_df['log_media_volume'] - log_mv_min) / (log_mv_max - log_mv_min)

# Derive Divergence_Score for both sets
train_df['Divergence_Score'] = train_si_norm - train_log_mv_norm
test_df['Divergence_Score'] = test_si_norm - test_log_mv_norm

In [11]:
# Separate features and target for training
y_train = train_df['Divergence_Score']
X_train = train_df.drop(columns=['Divergence_Score'])

# Separate features and target for testing
y_test = test_df['Divergence_Score']
X_test = test_df.drop(columns=['Divergence_Score'])

In [12]:
round(y_train.min(),4),y_train.max()

(-0.5514, 1.0)

#### 🔄 1.1.3 Missing Values & Outliers Handling 

Data is Clean and all the null values are handled in previously built local version of model

#### ⚙️ 1.1.4 Feature Engineering 
####  **Feature Generation** 
##### As mentioned above we need to create features

- Search_Velocity_Lag1
- Search_Velocity_Lag2

#### Calculating Search_Velocity_Lag1, Search_Velocity_Lag2

In [13]:
# calculating lag features
df['search_velocity_lag1'] = df.groupby(['country_name','category'])['search_velocity'].shift(1)
df['search_velocity_lag2'] = df.groupby(['country_name','category'])['search_velocity'].shift(2)

In [14]:
# merging the calculated features back to X_train
# Split into train and test
train_xdf = df[~df['year_month'].isin(last_6_months)]
X_train['search_velocity_lag1'] = train_xdf['search_velocity_lag1']
X_train['search_velocity_lag2'] = train_xdf['search_velocity_lag2']

In [15]:
# merging the calculated features back to X_test
test_xdf = df[df['year_month'].isin(last_6_months)]
X_test['search_velocity_lag1'] = test_xdf['search_velocity_lag1']
X_test['search_velocity_lag2'] = test_xdf['search_velocity_lag2']

In [16]:
X_train.columns[X_train.isnull().sum()>0]

Index(['search_velocity_lag1', 'search_velocity_lag2'], dtype='object')

In [17]:
X_test.columns[X_test.isnull().sum()>0]

Index([], dtype='object')

In [18]:
X_train.isnull().sum()

week_start                  0
country_name                0
category                    0
search_interest             0
search_velocity             0
search_acceleration         0
media_volume                0
tone_net_sentiment          0
tone_positive_score         0
tone_negative_score         0
tone_polarity               0
tone_activity_density       0
tone_self_group_density     0
source_diversity            0
demand_to_hype_ratio        0
inflation_rate              0
internet_penetration        0
gdp_per_capita              0
is_holiday                  0
holiday_count               0
year_month                  0
log_media_volume            0
search_velocity_lag1       42
search_velocity_lag2       84
dtype: int64

In [20]:
set(X_train['week_start'][X_train['search_velocity_lag2'].isnull()].to_list())

{Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-08 00:00:00')}

In [24]:
set(X_train['week_start'][X_train['search_velocity_lag1'].isnull()].to_list())

set()

### Removing columns with null values

In [25]:
X_train = X_train.dropna()

In [26]:
X_train.columns[X_train.isnull().sum()>0]

Index([], dtype='object')

In [27]:
X_train.shape

(6846, 24)

In [28]:
# Align y_train to the remaining indices
y_train = y_train.loc[X_train.index]

print(len(X_train), len(y_train)) 

6846 6846


### Created 2 feautes
- Search_Velocity_Lag1
- Search_Velocity_Lag2
### null values also handled

**Feature selection**

In [29]:
cols_to_remove=['week_start','country_code','year_month', 'search_interest', 'media_volume' ,'tone_positive_score', 'tone_negative_score','tone_self_group_density', 'demand_to_hype_ratio', 'media_volume_per_source','interest_per_source', 'gdp_per_capita', 'year', 'is_holiday']


In [30]:
X_train = X_train.drop(columns=cols_to_remove,axis=1,errors='ignore')

In [31]:
X_test = X_test.drop(columns=cols_to_remove,axis=1,errors='ignore')

In [32]:
print("X_train Columns",X_train.columns)
print("X_test Columns", X_test.columns)

X_train Columns Index(['country_name', 'category', 'search_velocity', 'search_acceleration',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'source_diversity', 'inflation_rate', 'internet_penetration',
       'holiday_count', 'log_media_volume', 'search_velocity_lag1',
       'search_velocity_lag2'],
      dtype='object')
X_test Columns Index(['country_name', 'category', 'search_velocity', 'search_acceleration',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'source_diversity', 'inflation_rate', 'internet_penetration',
       'holiday_count', 'log_media_volume', 'search_velocity_lag1',
       'search_velocity_lag2'],
      dtype='object')


#### Removed unnecessary columns from x_columns

### **Feature Modification:**
### **Encoding**

#### Categorical columns in dataset - Country_Name, Category - both are nominal columns

In [33]:
# Identify categorical columns from X_train
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# Initialize OneHotEncoder 
# handle_unknown='ignore' prevents errors on new categories in X_test
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on X_train categories and transform both X_train and X_test
train_encoded = encoder.fit_transform(X_train[categorical_cols])
test_encoded = encoder.transform(X_test[categorical_cols])

# Generate column names for the encoded features
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

# Convert encoded arrays back to DataFrames with original indices
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_feature_names, index=X_train.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_feature_names, index=X_test.index)

# Drop original categorical columns and join the new encoded columns
X_train = X_train.drop(columns=categorical_cols).join(train_encoded_df)
X_test = X_test.drop(columns=categorical_cols).join(test_encoded_df)

In [34]:
X_train.describe()

,search_velocity,search_acceleration,tone_net_sentiment,tone_polarity,tone_activity_density,source_diversity,inflation_rate,internet_penetration,holiday_count,log_media_volume,...,country_name_Mexico,country_name_Nigeria,country_name_Singapore,country_name_South_Africa,country_name_United_Arab_Emirates,country_name_United_Kingdom,country_name_United_States,category_Fashion_Beauty,category_Fitness_Wearables,category_Nutrition_Diets
count,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,...,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000,6846.000000
mean,0.082344,-0.050510,-0.260492,7.071957,20.940605,803.357143,5.565372,81.164768,0.220859,5.509324,...,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.333333,0.333333,0.333333
std,4.379539,6.346067,1.713279,1.068822,1.967881,1780.021796,7.205271,20.076305,0.567447,2.724643,...,0.257558,0.257558,0.257558,0.257558,0.257558,0.257558,0.257558,0.471439,0.471439,0.471439
min,-51.680000,-90.690000,-9.980800,1.162790,11.392400,0.000000,0.220000,32.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-1.960000,-2.890000,-1.452022,6.401555,19.945400,25.000000,2.390000,78.000000,0.000000,3.637586,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-0.100000,0.130000,-0.338512,7.100025,20.996000,134.000000,4.360000,90.000000,0.000000,5.499213,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.910000,3.037500,0.707877,7.705235,22.155700,628.750000,4.950000,95.000000,0.000000,7.487734,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
max,42.020000,38.780000,10.319000,13.924100,31.606500,12317.000000,33.240000,100.000000,6.000000,12.036843,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [37]:
# Drop non-numeric/identifier columns that shouldn't be scaled (like log_media_volumne, Country)
cols_to_scale = [
    "search_velocity",
    "search_acceleration",
    "tone_net_sentiment",
    "tone_polarity",
    "tone_activity_density",
    "source_diversity",
    "inflation_rate",
    "internet_penetration",
    "holiday_count",
    "search_velocity_lag1",
    "search_velocity_lag2"
]

# Check whether all required columns exist
missing_cols = [col for col in cols_to_scale if col not in X_train.columns]

print("Missing columns:", missing_cols)

# Keep only columns actually available in BOTH train and test
cols_to_scale = [
    col for col in cols_to_scale
    if col in X_train.columns and col in X_test.columns
]

# Make explicit copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train.loc[:, cols_to_scale] = scaler.fit_transform(
    X_train[cols_to_scale]
)

# Use the already-fitted scaler on test data
X_test.loc[:, cols_to_scale] = scaler.transform(
    X_test[cols_to_scale]
)

Missing columns: []


In [38]:
print(" Null values in X_test:",X_test.columns[X_test.isnull().sum()>0])
print(" NUll values in X_train:",X_train.columns[X_train.isnull().sum()>0])

 Null values in X_test: Index([], dtype='object')
 NUll values in X_train: Index([], dtype='object')


In [49]:
X_train.columns

Index(['search_velocity', 'search_acceleration', 'tone_net_sentiment',
       'tone_polarity', 'tone_activity_density', 'source_diversity',
       'inflation_rate', 'internet_penetration', 'holiday_count',
       'log_media_volume', 'search_velocity_lag1', 'search_velocity_lag2',
       'country_name_Australia', 'country_name_Brazil', 'country_name_Canada',
       'country_name_China', 'country_name_France', 'country_name_India',
       'country_name_Kenya', 'country_name_Mexico', 'country_name_Nigeria',
       'country_name_Singapore', 'country_name_South_Africa',
       'country_name_United_Arab_Emirates', 'country_name_United_Kingdom',
       'country_name_United_States', 'category_Fashion_Beauty',
       'category_Fitness_Wearables', 'category_Nutrition_Diets'],
      dtype='object')

#### 🤖 1.2 Model + Define, Train & Study

#### **Regression:**

In [41]:
# Define all regression models
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (Degree=2)": Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('linear', LinearRegression())
    ]),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Heuristic to determine fit status
    if train_r2 < 0.1 and test_r2 < 0.1:
        fit_status = "Underfit"
    elif (train_r2 - test_r2) > 0.15:
        fit_status = "Overfit"
    else:
        fit_status = "Goodfit"
        
    results.append({
        "Models": name,
        "TrainLoss (RMSE)": round(train_rmse, 2),
        "TestLoss (RMSE)": round(test_rmse, 2),
        "TrainScore (R²)": round(train_r2, 2),
        "TestScore (R²)": round(test_r2, 2),
        "Bias-Variance (Fit)": fit_status
    })

results_df = pd.DataFrame(results)

In [42]:
print("Final Evalaution Table")
display(results_df)

Final Evalaution Table


,Models,TrainLoss (RMSE),TestLoss (RMSE),TrainScore (R²),TestScore (R²),Bias-Variance (Fit)
0,Linear Regression,0.10,0.16,0.89,0.66,Overfit
1,Polynomial Regression (Degree=2),0.06,0.12,0.96,0.80,Overfit
2,Ridge,0.10,0.16,0.89,0.66,Overfit
3,Lasso,0.14,0.16,0.80,0.65,Goodfit
4,KNN Regressor,0.06,0.14,0.96,0.75,Overfit
5,SVR,0.07,0.11,0.95,0.84,Goodfit
6,Decision Tree,0.00,0.14,1.00,0.74,Overfit
7,Random Forest,0.02,0.12,0.99,0.82,Overfit
8,XGBoost,0.04,0.12,0.98,0.83,Overfit


In [39]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor


# Define the hyperparameter search space
param_distributions = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6,7,8],
    'min_child_weight': [3, 5, 10, 15],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 3.0, 5.0]
}

# Initialize base model
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

# Set up RandomizedSearchCV to explore the parameter space efficiently
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=30,             # Number of parameter combinations to try
    scoring='r2',
    cv=5,                  # 5-fold cross-validation on the training set
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the random search to the training data
random_search.fit(X_train, y_train)

# Extract the best model
xgb = random_search.best_estimator_

print("Best Parameters Found:")
for param, val in random_search.best_params_.items():
    print(f"  {param}: {val}")

# Evaluate best model on train and test sets
y_train_pred = xgb.predict(X_train)
y_test_pred = xgb.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

if train_r2 < 0.1 and test_r2 < 0.1:
    fit_status = "Underfit"
elif (train_r2 - test_r2) > 0.15:  
    fit_status = "Overfit"
else:
    fit_status = "Goodfit"

optimized_result = {
    "Models": "XGBoost (RandomSearch Optimized)",
    "TrainLoss (RMSE)": round(train_rmse, 2),
    "TestLoss (RMSE)": round(test_rmse, 2),
    "TrainScore (R²)": round(train_r2, 2),
    "TestScore (R²)": round(test_r2, 2),
    "Bias-Variance (Fit)": fit_status
}

print("\nFinal Evaluation:")
print(pd.DataFrame([optimized_result]).to_string(index=False))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters Found:
  subsample: 0.9
  reg_lambda: 1.0
  reg_alpha: 0.1
  n_estimators: 300
  min_child_weight: 10
  max_depth: 6
  learning_rate: 0.05
  colsample_bytree: 0.6

Final Evaluation:
                          Models  TrainLoss (RMSE)  TestLoss (RMSE)  TrainScore (R²)  TestScore (R²) Bias-Variance (Fit)
XGBoost (RandomSearch Optimized)              0.04             0.11             0.98            0.84             Goodfit


In [40]:
# Finalised XGBoost model and building model
xgb_tuned = XGBRegressor(
    n_estimators=300,            # More trees with a lower learning rate for stable generalization
    learning_rate=0.05,          # Slower learning rate prevents rapid memorization of training data
    max_depth=6,                 # Shallow trees prevent complex local decision boundaries
    min_child_weight=10,          # Requires more samples per leaf, stopping micro-splits
    subsample=0.9,               # Samples 90% of rows per tree to reduce variance
    colsample_bytree=0.8,        # Samples 80% of features per tree to break feature dominance
    reg_alpha=0.1,               # L1 regularization (Lasso penalty on weights)
    reg_lambda=1.0,              # L2 regularization (Ridge penalty on weights)
    random_state=42,
    n_jobs=-1
)

# Fit on training data
xgb_tuned.fit(X_train, y_train)

# Predict on train and test sets
y_train_pred = xgb_tuned.predict(X_train)
y_test_pred = xgb_tuned.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Determine fit status
if train_r2 < 0.1 and test_r2 < 0.1:
    fit_status = "Underfit"
elif (train_r2 - test_r2) > 0.15:
    fit_status = "Overfit"
else:
    fit_status = "Goodfit"

# Output final evaluation dictionary
tuned_result = {
    "Models": "XGBoost (Tuned)",
    "TrainLoss (RMSE)": round(train_rmse, 2),
    "TestLoss (RMSE)": round(test_rmse, 2),
    "TrainScore (R²)": round(train_r2, 2),
    "TestScore (R²)": round(test_r2, 2),
    "Bias-Variance (Fit)": fit_status
}

print(pd.DataFrame([tuned_result]).to_string(index=False))

         Models  TrainLoss (RMSE)  TestLoss (RMSE)  TrainScore (R²)  TestScore (R²) Bias-Variance (Fit)
XGBoost (Tuned)              0.04             0.11             0.98            0.84             Goodfit


#### 📥 1.5 Saving model & Real Time Prediction

In [48]:
import io
import os
import joblib
import boto3
from botocore.config import Config
from dotenv import load_dotenv

# ------------------------------------------------------------------
# 1. Load Environment Variables & Configure R2 Client
# ------------------------------------------------------------------
load_dotenv(override=True)

R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID", "").strip()
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID", "").strip()
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY", "").strip()
R2_BUCKET_NAME = os.getenv("R2_BUCKET_NAME", "").strip().strip("/")

if not all([R2_ACCOUNT_ID, R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_BUCKET_NAME]):
    raise ValueError("❌ Missing R2 environment variables in .env file.")

endpoint_url = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
    config=Config(
        signature_version="s3v4"
    )
)

def upload_object_to_r2(obj, r2_key: str) -> bool:
    """Serializes a Python object with joblib and uploads it directly to R2."""
    clean_key = r2_key.lstrip("/")
    try:
        print(f"⏳ Serializing '{clean_key}'...", end="", flush=True)
        buffer = io.BytesIO()
        joblib.dump(obj, buffer)
        payload_bytes = buffer.getvalue()
        size_kb = len(payload_bytes) / 1024
        print(f" ✅ Done ({size_kb:.2f} KB)")

        print(f"⏳ Uploading to R2 bucket '{R2_BUCKET_NAME}'...", end="", flush=True)
        response = s3_client.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=clean_key,
            Body=payload_bytes,
            ContentType="application/octet-stream"
        )
        
        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status_code == 200:
            print(f" ✅ Success! (HTTP 200) -> {R2_BUCKET_NAME}/{clean_key}")
            return True
        else:
            print(f" ⚠️ Unexpected HTTP status: {status_code}")
            return False
    except Exception as e:
        print(f"\n❌ Upload failed for '{clean_key}': {e}")
        return False

# ------------------------------------------------------------------
# 2. Local Save & R2 Upload Pipeline
# ------------------------------------------------------------------
folder_path = r"E:\ML Project Internship\ML-Project\app\pickles"
os.makedirs(folder_path, exist_ok=True)

artifacts = [
    ("xgb_tuned", xgb_tuned, "xgb_divergence_model.pkl"),
    ("encoder", encoder, "onehot_encoder.pkl"),
    ("scaler", scaler, "scaler.pkl"),
]

for name, obj, filename in artifacts:
    local_filepath = os.path.join(folder_path, filename)
    r2_key = f"pickles/{filename}"

    # 1. Save Locally
    try:
        joblib.dump(obj, local_filepath)
        print(f"✅ Saved local copy to: {local_filepath}")
    except Exception as e:
        print(f"❌ Error saving local {name}: {e}")

    # 2. Upload to Cloudflare R2
    upload_object_to_r2(obj, r2_key)
    print("-" * 60)

✅ Saved local copy to: E:\ML Project Internship\ML-Project\app\pickles\xgb_divergence_model.pkl
⏳ Serializing 'pickles/xgb_divergence_model.pkl'... ✅ Done (1012.09 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/pickles/xgb_divergence_model.pkl
------------------------------------------------------------
✅ Saved local copy to: E:\ML Project Internship\ML-Project\app\pickles\onehot_encoder.pkl
⏳ Serializing 'pickles/onehot_encoder.pkl'... ✅ Done (1.37 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/pickles/onehot_encoder.pkl
------------------------------------------------------------
✅ Saved local copy to: E:\ML Project Internship\ML-Project\app\pickles\scaler.pkl
⏳ Serializing 'pickles/scaler.pkl'... ✅ Done (1.31 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/pickles/scaler.pkl
--------------------------------------

## 📂 Saved Model and Preprocessing Steps (App Folder)

All essential components for deployment have been saved in the `app/pickles` directory:

| Component        | File Name                  | Purpose                                      |
|------------------|----------------------------|----------------------------------------------|
| XGBoost Model    | `xgb_divergence_model.pkl` | Final tuned model for predictions             |
| OneHot Encoder   | `onehot_encoder.pkl`       | Handles categorical feature transformation    |
| Standard Scaler  | `scaler.pkl`               | Normalizes numeric features for consistency   |

✅ These files ensure that both the **model** and the **preprocessing pipeline** can be reloaded seamlessly for inference.
